# DACON ETRI post-ID342 self-supervised subject-state runner

No-submit Kaggle runner for the approved non-DACON workflow. It stages the project, runs the bounded raw-stream subject-state smoke, runs the raw subject-state feature gate, runs the S3/Q3 raw sequence representation gate, and writes reports only. It does not create submission CSVs, base prediction artifacts, submission-log rows, or DACON submissions.

In [ ]:
import json
import os
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path


def run(cmd: str, cwd: Path | None = None) -> None:
    print(f"\n$ {cmd}", flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONHASHSEED", "0")
    env.setdefault("OMP_NUM_THREADS", "2")
    env.setdefault("OPENBLAS_NUM_THREADS", "2")
    env.setdefault("MKL_NUM_THREADS", "2")
    env.setdefault("NUMEXPR_NUM_THREADS", "2")
    if cwd is not None:
        env["PYTHONPATH"] = str(cwd)
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd) if cwd else None, env=env)


is_kaggle = Path("/kaggle/input").exists()
input_root = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input" if is_kaggle else "."))
working_root = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working" if is_kaggle else "build/kaggle_self_supervised_subject_state_smoke"))
scratch_root = Path(os.environ.get("KAGGLE_SCRATCH_DIR", "/tmp/dacon_etri_ss_subject_state" if is_kaggle else str(working_root / "scratch")))
project_dir = scratch_root / "project"
output_dir = working_root / "kaggle_outputs"
output_dir.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLE_OUTPUT_DIR"] = str(output_dir)

if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True, exist_ok=True)

copy_names = ["src", "conf", "scripts", "data", "artifacts", "submissions", "reports", "pyproject.toml", "README.md", "dataset-metadata.json", "Makefile"]
patch_script_names = [
    "run_post_id342_self_supervised_subject_state_no_submit.py",
    "diagnose_post_id342_raw_subject_state_feature_gate.py",
    "diagnose_post_id342_s3q3_raw_sequence_representation_gate.py",
]
runner_script_name = patch_script_names[0]
feature_gate_script_name = patch_script_names[1]
sequence_gate_script_name = patch_script_names[2]


def overlay_project(source_root: Path, dst_root: Path) -> None:
    source_zip = source_root / "source.zip"
    if source_zip.exists():
        tmp = scratch_root / f"unzipped_{source_root.name}"
        if tmp.exists():
            shutil.rmtree(tmp)
        tmp.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source_zip) as zf:
            zf.extractall(tmp)
        source_root = tmp
    for name in copy_names:
        src = source_root / name
        if not src.exists():
            continue
        dst = dst_root / name
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    for script_name in patch_script_names:
        standalone_script = source_root / script_name
        if standalone_script.exists():
            dst = dst_root / "scripts" / script_name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(standalone_script, dst)


def add_candidate_root(candidate_roots: set[Path], path: Path, ancestor_index: int = 0) -> None:
    if len(path.parents) <= ancestor_index:
        return
    candidate_roots.add(path.parents[ancestor_index])


candidate_roots: set[Path] = set()
if input_root.exists():
    for source_zip in input_root.rglob("source.zip"):
        candidate_roots.add(source_zip.parent)
    for script_name in patch_script_names:
        for marker in input_root.rglob(script_name):
            if marker.parent.name == "scripts":
                add_candidate_root(candidate_roots, marker, 1)
            else:
                candidate_roots.add(marker.parent)
    for marker in input_root.rglob("pyproject.toml"):
        candidate_roots.add(marker.parent)
    for raw_file in input_root.rglob("data/raw/ch2025_data_items/ch2025_mGps.parquet"):
        add_candidate_root(candidate_roots, raw_file, 3)

def root_priority(path: Path) -> tuple[int, str]:
    text = str(path)
    if "dacon-etri-human-base" in text:
        return (0, text)
    if "dacon-etri-upstream-patch-post-id341" in text:
        return (1, text)
    if "dacon-etri-ss-subject-state-patch-id342" in text:
        return (2, text)
    return (1, text)


roots = sorted(candidate_roots, key=root_priority)
if not roots:
    roots = [Path.cwd()]
for root in roots:
    print(f"Overlaying input root: {root}")
    overlay_project(root, project_dir)

required = [
    project_dir / "scripts/run_post_id342_self_supervised_subject_state_no_submit.py",
    project_dir / "scripts/diagnose_post_id342_raw_subject_state_feature_gate.py",
    project_dir / "scripts/diagnose_post_id342_s3q3_raw_sequence_representation_gate.py",
    project_dir / "data/raw/ch2025_data_items/ch2025_mGps.parquet",
    project_dir / "data/raw/ch2025_data_items/ch2025_wHr.parquet",
]
missing = [str(path.relative_to(project_dir)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Packaged project is missing required paths: {missing}")

max_rows = int(os.environ.get("MAX_ROWS_PER_MODALITY", "200000"))
token_limit = int(os.environ.get("TOKEN_LIMIT", "32768"))
epochs = int(os.environ.get("SMOKE_EPOCHS", "3"))
cmd = " ".join([
    "python scripts/run_post_id342_self_supervised_subject_state_no_submit.py",
    "--mode remote_no_submit",
    f"--max-rows-per-modality {max_rows}",
    "--batch-size 65536",
    f"--token-limit {token_limit}",
    f"--epochs {epochs}",
])
run(cmd, cwd=project_dir)

gate_max_rows = int(os.environ.get("GATE_MAX_ROWS_PER_MODALITY", "0"))
gate_output = output_dir / "remote_post_id342_raw_subject_state_feature_gate_20260624.json"
gate_cmd = " ".join([
    "python scripts/diagnose_post_id342_raw_subject_state_feature_gate.py",
    f"--max-rows-per-modality {gate_max_rows}",
    "--batch-size 65536",
    f"--output {gate_output}",
])
run(gate_cmd, cwd=project_dir)

sequence_n_kernels = int(os.environ.get("SEQUENCE_REP_N_KERNELS", "256"))
sequence_seeds = os.environ.get("SEQUENCE_REP_SEEDS", "45502,45602")
sequence_families = os.environ.get("SEQUENCE_REP_FAMILIES", "all")
sequence_gate_output = output_dir / "remote_post_id342_s3q3_raw_sequence_representation_gate_20260624.json"
sequence_gate_cmd = " ".join([
    "python scripts/diagnose_post_id342_s3q3_raw_sequence_representation_gate.py",
    f"--n-kernels {sequence_n_kernels}",
    f"--seeds {sequence_seeds}",
    f"--families {sequence_families}",
    f"--output {sequence_gate_output}",
])
run(sequence_gate_cmd, cwd=project_dir)
if is_kaggle and project_dir.exists():
    shutil.rmtree(project_dir)

suspicious = []
for path in output_dir.rglob("*"):
    if not path.is_file():
        continue
    lower = path.name.lower()
    if path.suffix.lower() == ".csv" or "submission" in lower or "base_oof" in lower or "base_test" in lower:
        suspicious.append(str(path))
if suspicious:
    raise RuntimeError(f"Forbidden submission-like output detected: {suspicious}")

manifest = {
    "created_at_kst": datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M:%S UTC+09:00"),
    "runner": "post_id342_self_supervised_subject_state_no_submit",
    "mode": "remote_no_submit",
    "max_rows_per_modality": max_rows,
    "token_limit": token_limit,
    "epochs": epochs,
    "feature_gate_max_rows_per_modality": gate_max_rows,
    "feature_gate_output": str(gate_output.relative_to(output_dir)),
    "sequence_representation_n_kernels": sequence_n_kernels,
    "sequence_representation_seeds": sequence_seeds,
    "sequence_representation_families": sequence_families,
    "sequence_representation_gate_output": str(sequence_gate_output.relative_to(output_dir)),
    "output_files": [str(path.relative_to(output_dir)) for path in sorted(output_dir.rglob("*")) if path.is_file()],
    "no_submission_created": True,
    "no_submission_artifact_created": True,
    "no_submission_log_row_created": True,
    "no_dacon_submission_made": True,
}
(output_dir / "post_id342_self_supervised_subject_state_runner_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps(manifest, indent=2, ensure_ascii=False))
